# Evidencia de Aprendizaje 1: Carga y Transformación de Dataset
**Objetivo:** Conectar con Hugging Face, cargar un dataset como DataFrame de pandas, aplicar cuatro transformaciones distintas y generar un reporte automático.

### Importación y configuración del Path

In [1]:
import sys
from pathlib import Path

# Configurar la ruta raíz para importar las clases desde src/clases_po
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader, Reportes

/home/runner/work/AD_2026Q4/AD_2026Q4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Carga del Dataset

In [2]:
# Carga del dataset original
loader = HFDatasetLoader(
    dataset_name="JasonZhouTI/Fast_Food_Restaurants",
    split="train",
    filename="datos/fast_food_restaurants.csv",
)
df = loader.load()

print(f"Dimensiones del DataFrame original: {df.shape}")
df.head(3)

Repo card metadata block was not found. Setting CardData to empty.


Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 10000/10000 [00:00<00:00, 770586.81 examples/s]

Dimensiones del DataFrame original: (10000, 10)


,address,city,country,keys,latitude,longitude,name,postalCode,province,websites
0,324 Main St,Massena,US,us/ny/massena/324mainst/-1161002137,44.92130,-74.89021,McDonald's,13662,NY,"http://mcdonalds.com,http://www.mcdonalds.com/..."
1,530 Clinton Ave,Washington Court House,US,us/oh/washingtoncourthouse/530clintonave/-7914...,39.53255,-83.44526,Wendy's,43160,OH,http://www.wendys.com
2,408 Market Square Dr,Maysville,US,us/ky/maysville/408marketsquaredr/1051460804,38.62736,-83.79141,Frisch's Big Boy,41056,KY,"http://www.frischs.com,https://www.frischs.com..."


### Aplicación de las 4 Transformaciones requeridas
1. **Limpieza:** Eliminación de valores nulos.
2. **Estructura:** Renombrado de columnas.
3. **Filtrado:** Selección por condición lógica.
4. **Organización:** Selección de columnas específicas y ordenamiento.

### Transformaciones 1 y 2

In [3]:
# Transformación 1: Eliminar registros que tengan valores nulos en la columna 'websites'
df_t1 = df.dropna(subset=["websites"]).copy()
print(f"Filas tras eliminar nulos: {len(df_t1)}")

# Transformación 2: Renombrar columnas al español
df_t2 = df_t1.rename(columns={
    "name": "restaurante",
    "address": "direccion",
    "city": "ciudad",
    "province": "estado",
    "postalCode": "codigo_postal"
})
df_t2.head(3)

Filas tras eliminar nulos: 9535


,direccion,ciudad,country,keys,latitude,longitude,restaurante,codigo_postal,estado,websites
0,324 Main St,Massena,US,us/ny/massena/324mainst/-1161002137,44.92130,-74.89021,McDonald's,13662,NY,"http://mcdonalds.com,http://www.mcdonalds.com/..."
1,530 Clinton Ave,Washington Court House,US,us/oh/washingtoncourthouse/530clintonave/-7914...,39.53255,-83.44526,Wendy's,43160,OH,http://www.wendys.com
2,408 Market Square Dr,Maysville,US,us/ky/maysville/408marketsquaredr/1051460804,38.62736,-83.79141,Frisch's Big Boy,41056,KY,"http://www.frischs.com,https://www.frischs.com..."


### Transformaciones 3 y 4

In [4]:
# Transformación 3: Filtrar restaurantes ubicados en Nueva York (NY) y Ohio (OH)
df_t3 = df_t2[df_t2["estado"].isin(["NY", "OH"])].copy()
print(f"Registros en NY y OH: {len(df_t3)}")

# Transformación 4: Seleccionar columnas clave y ordenar alfabéticamente por ciudad
columnas_finales = ["restaurante", "direccion", "ciudad", "estado", "codigo_postal", "latitude", "longitude"]
df_transformado = df_t3[columnas_finales].sort_values(by=["ciudad", "restaurante"]).reset_index(drop=True)
df_transformado.head()

Registros en NY y OH: 775


,restaurante,direccion,ciudad,estado,codigo_postal,latitude,longitude
0,Boston Market,3750 W Market St Unit F,Akron,OH,44333,41.135500,-81.634000
1,McDonald's,1810 Merriman Rd,Akron,OH,44313,41.135107,-81.554496
2,McDonalds,2021 E Market St,Akron,OH,44312,41.060767,-81.455031
3,Wendy's,1266 Canton Rd,Akron,OH,44312,41.030490,-81.427730
4,Arby's,1 Crossgates Mall Rd,Albany,NY,12203,42.687711,-73.849772


### Generación del Reporte Final

In [5]:
# Generar y guardar el reporte del DataFrame transformado
reportes = Reportes()
ruta_reporte = reportes.guardar_reporte(
    df_transformado,
    "reportes/resumen_transformado.txt",
)

# Imprimir en consola para verificación
reportes.imprimir_reporte(df_transformado)
print(f"\n✅ Reporte guardado exitosamente en: {ruta_reporte}")

REPORTE DEL DATAFRAME
Filas: 775
Columnas: 7
Duplicados: 0

Columnas y tipos:
- restaurante: str
- direccion: str
- ciudad: str
- estado: str
- codigo_postal: str
- latitude: float64
- longitude: float64

Valores nulos por columna:
- restaurante: 0
- direccion: 0
- ciudad: 0
- estado: 0
- codigo_postal: 0
- latitude: 0
- longitude: 0

Estadísticas numéricas:
       latitude  longitude
count   775.000    775.000
mean     40.676    -80.353
std       1.386      7.132
min      30.400    -87.683
25%      39.397    -84.253
50%      40.710    -82.067
75%      41.415    -76.028
max      44.950     81.188

✅ Reporte guardado exitosamente en: reportes/resumen_transformado.txt
